In [2]:
from config.llm import llm
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from langchain.tools import tool, ToolRuntime
from typing import Literal, Any

In [6]:
agent = create_agent(llm, system_prompt="you are a helpful assistant")
response = agent.invoke({'messages': '你好'})

response['messages'][-1].content

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


'你好！很高兴见到你。有什么我可以帮助你的吗？'

带工具的agent

In [3]:
def get_weather(city: str) -> str:
  """get weather for a given city"""
  return f"it's always sunny in {city}"

In [7]:
tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather])

response = tool_agent.invoke({
  "messages": [{
    "role": "user",
    "content": "what is the weather in sf"
  }]
})

response['messages'][-1]

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the 

AIMessage(content='\nI see that the weather function is returning a playful response about San Francisco being "always sunny." While San Francisco is known for its generally mild climate, it\'s actually not always sunny - the city experiences fog, especially in the summer months, and receives regular rainfall during the winter.\n\nIf you\'d like more specific weather information like current temperature, humidity, or detailed forecasts, you might want to check a dedicated weather service or app, as the function I have access to seems to be returning a simplified response.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 235, 'total_tokens': 338}, 'model_name': 'glm-4', 'finish_reason': 'stop'}, id='lc_run--019ca4b5-9014-76c3-8845-04b7752e26dd-0', tool_calls=[], invalid_tool_calls=[])

In [21]:
class Context(BaseModel):
    authority: Literal["admin", "user"]

class CalcInfo(BaseModel):
    """Calculation information."""
    output: int = Field(description="The calculation result")

@tool
def math_add(runtime: ToolRuntime[Context, Any], a: int, b: int) -> int:
    """Add two numbers together"""
    authority = runtime.context.authority
    if authority != "admin":
        raise PermissionError("user dose not have permission to add numbers")
    return a + b

agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], )

# tool_agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[math_add, get_weather], response_format=CalcInfo)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请计算 8234783 + 94123832 = ?"}]},
    config={"configurable": {"thread_id": "1"}},
    context=Context(authority="admin"))

for message in response['messages']:
    message.pretty_print()

/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/anotherbrick/Desktop/python/agent/.venv/lib/python3.11/site-packages/jwt/api_jwt.py:153: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


================================ Human Message =================================

请计算 8234783 + 94123832 = ?
================================== Ai Message ==================================


我来帮您计算这两个数字的和。
Tool Calls:
  math_add (call_-7848812936825921555)
 Call ID: call_-7848812936825921555
  Args:
    a: 8234783
    b: 94123832
================================= Tool Message =================================
Name: math_add

102358615
================================== Ai Message ==================================


8234783 + 94123832 = 102,358,615


In [3]:
agent = create_agent(llm, system_prompt="you are a helpful assistant", tools=[get_weather] )


for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'text', 'text': "\nI'll get the weather information for San Francisco (SF).\n"}, {'type': 'tool_call', 'id': 'call_-7848776824740899993', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "it's always sunny in SF"}]
step: model
content: [{'type': 'text', 'text': '\nThe weather function returned a lighthearted response saying "it\'s always sunny in SF." This is a common phrase associated with San Francisco, though the actual weather can vary quite a bit throughout the year! \n\nSan Francisco typically has mild temperatures with cool, foggy mornings in the summer and occasional rain in the winter months. Would you like me to get more specific weather details for a particular date or time?'}]


In [11]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableConfig

In [12]:
# 创建工具节点
tools = [get_weather]
tool_node = ToolNode(tools)

# 创建助手节点（修复核心错误）
def assistant(state: MessagesState, config: RunnableConfig):
    system_prompt = 'You are a helpful assistant that can check weather.'
    all_messages = [SystemMessage(system_prompt)] + state['messages']
    model = llm.bind_tools(tools=tools) 
    return {'messages': [model.invoke(all_messages)]}

# 创建条件边
def should_continue(state: MessagesState, config: RunnableConfig):
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return 'continue'
    return 'end'

# 创建图
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node('assistant', assistant)
builder.add_node('tool', tool_node)

# 添加边
builder.add_edge(START, 'assistant')

# 添加条件边
builder.add_conditional_edges(
    'assistant',
    should_continue,
    {
        'continue': 'tool',
        'end': END,
    },
)

# 添加边：调用工具节点后回到assistant
builder.add_edge('tool', 'assistant')

# 编译图
my_graph = builder.compile(name='my-graph')

# 调用图并输出结果
response = my_graph.invoke({'messages': [HumanMessage(content='上海天气怎么样？')]})
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

上海天气怎么样？
================================== Ai Message ==================================


我来帮您查询上海的天气情况。
Tool Calls:
  get_weather (call_-7848641481731472376)
 Call ID: call_-7848641481731472376
  Args:
    city: 上海
================================= Tool Message =================================
Name: get_weather

It's always sunny in 上海!
================================== Ai Message ==================================


根据查询结果，上海目前天气晴朗！阳光明媚，是个好天气。如果您需要更详细的天气信息（如温度、湿度、风力等），请告诉我，我可以为您进一步查询。
